In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer


# Get the same results each time
np.random.seed(0)


# Load the training data
data = pd.read_csv("data.csv")
comments = data["comment_text"]
target = (data["target"]>0.7).astype(int) # Преобразуем вероятности в бинарные метки: 1, если target > 0.7, иначе 0

In [2]:
data

,id,target,comment_text,severe_toxicity,obscene,identity_attack,insult,threat,asian,atheist,...,article_id,rating,funny,wow,sad,likes,disagree,sexual_explicit,identity_annotator_count,toxicity_annotator_count
0,59856,0.893617,haha you guys are a bunch of losers.,0.021277,0.000000,0.021277,0.872340,0.0000,0.0,0.0,...,2006,rejected,0,0,0,1,0,0.000000,4,47
1,239607,0.912500,Yet call out all Muslims for the acts of a few...,0.050000,0.237500,0.612500,0.887500,0.1125,0.0,0.0,...,26670,approved,0,0,0,1,0,0.000000,4,80
2,239612,0.830769,This bitch is nuts. Who would read a book by a...,0.107692,0.661538,0.338462,0.830769,0.0000,0.0,0.0,...,26674,rejected,0,0,0,0,0,0.061538,4,65
3,240311,0.968750,You're an idiot.,0.031250,0.062500,0.000000,0.968750,0.0000,NaN,NaN,...,32846,rejected,0,0,0,0,0,0.000000,0,32
4,240329,0.900000,Who cares!? Stark trek and Star Wars fans are ...,0.100000,0.200000,0.000000,0.900000,0.0000,NaN,NaN,...,32846,rejected,0,0,0,0,0,0.300000,0,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90897,957507,0.166667,Methinks Bishop Braxton doth protest too much ...,0.000000,0.000000,0.166667,0.166667,0.0000,0.0,0.0,...,166439,approved,0,0,0,6,0,0.000000,10,6
90898,5363032,0.000000,Sounds pretty speculative to me. But i'm a sp...,0.000000,0.000000,0.000000,0.000000,0.0000,NaN,NaN,...,340892,approved,3,0,0,1,0,0.000000,0,4
90899,5910478,0.166667,Seriously!\nVery proud of our 'domestic progra...,0.000000,0.000000,0.166667,0.000000,0.0000,NaN,NaN,...,374717,approved,0,0,0,0,1,0.000000,0,6
90900,1052094,0.000000,Hawaii food is mostly GMO loaded with chemical...,0.000000,0.000000,0.000000,0.000000,0.0000,NaN,NaN,...,315373,approved,0,0,0,0,1,0.000000,0,6


In [3]:
target

0        1
1        1
2        1
3        1
4        1
        ..
90897    0
90898    0
90899    0
90900    0
90901    0
Name: target, Length: 90902, dtype: int32

In [4]:
# Задание 1
# Разделение данных на обучающую и тестовую выборки (70% train, 30% test)
comments_train, comments_test, target_train, target_test = train_test_split(
    comments, target, test_size=0.3, random_state=0
)

comments_train.info()

<class 'pandas.core.series.Series'>
Index: 63631 entries, 77401 to 68268
Series name: comment_text
Non-Null Count  Dtype 
--------------  ----- 
63631 non-null  object
dtypes: object(1)
memory usage: 994.2+ KB


In [5]:
# Задание 2
# Преобразование текста в числовой формат с помощью CountVectorizer

vectorizer = CountVectorizer()

# Обучаем векторизатор и преобразуем
comments_train_vec = vectorizer.fit_transform(comments_train)

# Применяем обученный векторизатор 
comments_test_vec = vectorizer.transform(comments_test)

print(f"Размерность обучающей выборки после векторизации: {comments_train_vec.shape}")
print(f"Размерность тестовой выборки после векторизации: {comments_test_vec.shape}")


Размерность обучающей выборки после векторизации: (63631, 58152)
Размерность тестовой выборки после векторизации: (27271, 58152)


In [6]:
# Задание 3

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Создаем экземпляр модели LogisticRegression
# Увеличиваем max_iter для сходимости на больших данных
model = LogisticRegression(max_iter=2000)

# Обучаем модель на обучающих данных
model.fit(comments_train_vec, target_train)

# Делаем предсказания на тестовой выборке
predictions = model.predict(comments_test_vec)

# Считаем accuracy
accuracy = accuracy_score(target_test, predictions)

print(f"Accuracy модели на тестовой выборке: {accuracy:.4f}")


Accuracy модели на тестовой выборке: 0.9287


In [7]:
# Задание 4
# Функция для предсказания токсичности нового комментария
def predict_toxicity(comment: str) -> float:
    # Предсказывает вероятность того, что комментарий является токсичным.
    # Входной комментарий -строка
    # Вероятность токсичности - число от 0 до 1 (1-высшая токсичность)

    # Преобразуем входной комментарий с использованием того же обученного векторизатора
    comment_vec = vectorizer.transform([comment])

    # Получаем вероятности для каждого класса (0 и 1)
    probabilities = model.predict_proba(comment_vec)

    # Возвращаем вероятность для класса 1 (токсичный комментарий)
    return probabilities[0][1]

# для примера:
print(predict_toxicity("python it's crazy ass"))
print(predict_toxicity("python is good"))

0.998807126929604
0.12243276400613731


In [8]:
# Задание 5
# Тестирование функции на примерах комментариев
comment1 = "Apples are stupid"
comment2 = "I love apples"

print(predict_toxicity(comment1))
print(predict_toxicity(comment2))



0.999500390918607
0.09918969018065577


Как видим модель справляеться со своей задачей

In [10]:
# Задание 6 
# Вывод десяти слов с наибольшими коэффициентами (считающихся наиболее токсичными)

# Получаем коэффициенты 
word_coefficients = model.coef_[0]

# Создадим обратный словарь: индекс -> слово
index_to_word = {index: word for word, index in vectorizer.vocabulary_.items()}

# Сопоставляем коэффициенты словам, используя обратный словарь
word_to_coeff= {
    index_to_word[i]: word_coefficients[i]
    for i in range(len(word_coefficients)) # Проходим по всем индексам коэффициентов
}

# Сортируем слова по убыванию коэффициентов из альтернативного словаря и берем топ-10
most_toxic_words = sorted(word_to_coeff.items(), key=lambda item: item[1], reverse=True)[:20]

for word, coeff in most_toxic_words:
    print(f"Слово: '{word}' | Коэффициент: {coeff:.4f}")


Слово: 'stupid' | Коэффициент: 9.5828
Слово: 'idiot' | Коэффициент: 8.6971
Слово: 'idiots' | Коэффициент: 8.6424
Слово: 'stupidity' | Коэффициент: 7.6042
Слово: 'idiotic' | Коэффициент: 6.7946
Слово: 'crap' | Коэффициент: 6.5536
Слово: 'dumb' | Коэффициент: 6.5327
Слово: 'pathetic' | Коэффициент: 6.4639
Слово: 'moron' | Коэффициент: 6.3474
Слово: 'morons' | Коэффициент: 6.3326
Слово: 'hypocrite' | Коэффициент: 6.2434
Слово: 'ignorant' | Коэффициент: 6.1978
Слово: 'damn' | Коэффициент: 6.1249
Слово: 'fools' | Коэффициент: 6.1000
Слово: 'shit' | Коэффициент: 5.7060
Слово: 'jerk' | Коэффициент: 5.6488
Слово: 'ass' | Коэффициент: 5.6024
Слово: 'ridiculous' | Коэффициент: 5.5833
Слово: 'hypocrites' | Коэффициент: 5.5122
Слово: 'fool' | Коэффициент: 5.4252


# Задание 7
Присутствуют слова токсичные,  но некоторые надо рассматривать в  контексте:
pathetic - жалкий
hypocrite - лицимер
ignorant - невежественный
который модель не улавливает

In [12]:
# Для примера:
comment1 = "Bill is a hypocrite because he hides the truth" #Бил лицимер, потомучто скрывает правду
comment2 = "He was told constantly that he was ignorant" #Ему постоянно говорили, что он невежественный

print(predict_toxicity(comment1))
print(predict_toxicity(comment2))


0.9883888913093658
0.9883994470716678


In [13]:
# Задание 8
print(predict_toxicity("I have a christian friend"))
print(predict_toxicity("I have a muslim friend"))
print(predict_toxicity("I have a white friend"))
print(predict_toxicity("I have a black friend"))

0.12169796825169599
0.44551833474534047
0.32372491806557074
0.5137783711176015


Чувствуеться  bias в сторону масульман и чернокожих людей.

# Задание 9 
Это приводит к нарушению справедливости модели, так как ее предсказания зависят не только от реальной токсичности комментария, но и от его темы, если эта тема исторически или социально ассоциируется с негативными высказываниями в обучающем наборе данных (нергы, а также то что все терроисты ислам (Алькаида), тое сть предвзятость обучающего набора данных.

# Задание 10
- Собрать более сбалансированный датасет -  включать в обучающую выборку примеры нетоксичных комментариев на темы, которые в равной степени будут затргивать и религию и цвет кожи, а также токсичные комментарии, которые не затрагивают эти чувствительные темы.

- попробовать аугментировать данные

- ориентироватся не только на accuracy, но и на Precision Recall